# Unidade 3 — Aula 3
## Criação de novos atributos
Ciência de Dados · Bacharelado em Engenharia de Software · UNIFSA


**Situação:** o dono de um restaurante quer saber quais clientes dão as melhores gorjetas e como organizar o atendimento. As colunas originais não respondem diretamente a essas perguntas. Nesta atividade, você vai criar novos atributos a partir das colunas existentes.

**Base de dados:** *Tips* (BRYANT, P. G.; SMITH, M. A., 1995). 244 contas registradas por um garçom em um restaurante dos Estados Unidos, com valores em dólares. A base é carregada de uma cópia pública:
https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv


Aluno: Carlos Gabriel Raposo Landim

## 1. Carga e preparação da base

In [1]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
from pathlib import Path
df = pd.read_csv(Path('dados/tips.csv') if Path('dados/tips.csv').exists() else URL)
df.columns = ["conta", "gorjeta", "sexo", "fumante", "dia", "turno", "pessoas"]
df = df.drop_duplicates().reset_index(drop=True)
df.head()

,conta,gorjeta,sexo,fumante,dia,turno,pessoas
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


| Coluna | Descrição |
|---|---|
| conta | valor da conta, em dólares, sem a gorjeta |
| gorjeta | valor da gorjeta, em dólares |
| sexo | sexo de quem pagou a conta |
| fumante | se havia fumantes na mesa (Yes, No) |
| dia | dia da semana (Thur, Fri, Sat, Sun) |
| turno | almoço (Lunch) ou jantar (Dinner) |
| pessoas | número de pessoas na mesa |

## 2. Razão: dividir uma coluna por outra
**Exemplo:** percentual da gorjeta em relação à conta, e sua média por tamanho de grupo.

In [2]:
df["percentual_gorjeta"] = df["gorjeta"] / df["conta"] * 100
df.groupby("pessoas")[["gorjeta", "percentual_gorjeta"]].mean().round(2)

,gorjeta,percentual_gorjeta
pessoas,,
1,1.44,21.73
2,2.59,16.58
3,3.39,15.22
4,4.14,14.59
5,4.03,14.15
6,5.22,15.62


**Sua vez 2.1:** crie o atributo `conta_por_pessoa` (valor da conta dividido pelo número de pessoas) e calcule sua média por turno.

In [3]:
df['conta_por_pessoa'] = df['conta'] / df['pessoas']
df.groupby('turno')[['conta', 'conta_por_pessoa']].mean().round(2)

,conta,conta_por_pessoa
turno,,
Dinner,20.80,8.11
Lunch,17.23,7.33


**Pergunta 2.2:** em qual turno cada cliente gasta mais, individualmente? Que diferença existe entre olhar `conta` e olhar `conta_por_pessoa`?

**Resposta:**

No jantar, o gasto médio por pessoa é US$ 8,11; no almoço, US$ 7,33. A média da conta inteira também é maior no jantar (US$ 20,80 contra US$ 17,23), mas inclui o efeito do tamanho da mesa. Dividir por pessoas permite comparar o gasto individual entre mesas de tamanhos diferentes.

## 3. Soma: combinar colunas na mesma unidade
**Exemplo:** total efetivamente pago pelo cliente.

In [4]:
df["total_pago"] = df["conta"] + df["gorjeta"]
df[["conta", "gorjeta", "total_pago"]].head()

,conta,gorjeta,total_pago
0,16.99,1.01,18.00
1,10.34,1.66,12.00
2,21.01,3.50,24.51
3,23.68,3.31,26.99
4,24.59,3.61,28.20


**Sua vez 3.1:** crie o atributo `total_por_pessoa` (total pago dividido pelo número de pessoas) e mostre as cinco mesas com maior valor.

In [5]:
df['total_por_pessoa'] = df['total_pago'] / df['pessoas']
df.nlargest(5, 'total_por_pessoa')[['conta', 'gorjeta', 'pessoas', 'total_pago', 'total_por_pessoa']].round(2)

,conta,gorjeta,pessoas,total_pago,total_por_pessoa
184,40.55,3.00,2,43.55,21.78
170,50.81,10.00,3,60.81,20.27
179,34.63,3.55,2,38.18,19.09
83,32.68,5.00,2,37.68,18.84
175,32.90,3.11,2,36.01,18.00


## 4. Indicador: transformar uma condição em 0 ou 1
**Exemplo:** a mesa foi atendida no fim de semana? Comparação do percentual médio da gorjeta.

In [6]:
df["fim_de_semana"] = df["dia"].isin(["Sat", "Sun"]).astype(int)
df.groupby("fim_de_semana")["percentual_gorjeta"].agg(["count", "mean"]).round(2)

,count,mean
fim_de_semana,,
0,80,16.34
1,163,15.96


**Sua vez 4.1:** crie o indicador `grupo_grande`, igual a 1 para mesas com 4 pessoas ou mais e 0 nos demais casos. Compare o percentual médio da gorjeta entre os dois grupos.

In [7]:
df['grupo_grande'] = (df['pessoas'] >= 4).astype(int)
df.groupby('grupo_grande')['percentual_gorjeta'].agg(['count', 'mean']).round(2)

,count,mean
grupo_grande,,
0,197,16.42
1,46,14.64


**Pergunta 4.2:** o fim de semana e o tamanho do grupo influenciam o percentual da gorjeta da mesma forma? Justifique com os números.

**Resposta:**

Nesta amostra, os dois recortes têm percentual médio menor, mas a diferença é bem maior para grupos grandes: 14,64% em 46 mesas, contra 16,42% em 197 mesas menores (−1,79 ponto percentual). No fim de semana são 15,96% em 163 mesas, contra 16,34% em 80 mesas nos demais dias (−0,39 ponto). São associações descritivas; esses números não demonstram efeito causal do dia ou do tamanho do grupo.

## 5. Faixa: transformar um número em categorias ordenadas
**Exemplo:** faixas de valor da conta com cortes fixos (até 15, de 15 a 25, acima de 25 dólares).

In [8]:
df["faixa_conta"] = pd.cut(df["conta"], bins=[0, 15, 25, np.inf], labels=["baixa", "média", "alta"])
df.groupby("faixa_conta", observed=True)["percentual_gorjeta"].agg(["count", "mean"]).round(2)

,count,mean
faixa_conta,,
baixa,79,18.39
média,109,15.88
alta,55,13.17


**Sua vez 5.1:** crie o atributo `faixa_conta_igual` usando `pd.qcut`, que divide as mesas em três grupos com a mesma quantidade de mesas. Mostre o menor e o maior valor de conta de cada faixa e o percentual médio da gorjeta.

In [9]:
df['faixa_conta_igual'] = pd.qcut(df['conta'], q=3, labels=['baixa', 'média', 'alta'])
df.groupby('faixa_conta_igual', observed=True).agg(mesas=('conta', 'size'), conta_minima=('conta', 'min'), conta_maxima=('conta', 'max'), percentual_medio=('percentual_gorjeta', 'mean')).round(2)

,mesas,conta_minima,conta_maxima,percentual_medio
faixa_conta_igual,,,,
baixa,81,3.07,15.04,18.27
média,81,15.06,21.01,16.17
alta,81,21.16,50.81,13.81


**Pergunta 5.2:** qual das duas formas de criar faixas você apresentaria ao dono do restaurante? Por quê?

**Resposta:**

Apresentaria primeiro as faixas de cortes fixos: até US$ 15, acima de US$ 15 até US$ 25 e acima de US$ 25. São fáceis de explicar e mantêm o mesmo significado em semanas diferentes. O qcut ajuda a comparar grupos com cerca de um terço da amostra, mas seus limites dependem dos dados: aqui ficam próximos de US$ 15,05 e US$ 21,06. Empates nos valores podem impedir quantidades exatamente iguais em cada faixa.

## 6. Agregação: comparar cada mesa com o seu grupo
**Exemplo:** percentual médio da gorjeta do dia de cada mesa e indicador de mesa acima da média do seu dia.

In [10]:
df["media_do_dia"] = df.groupby("dia")["percentual_gorjeta"].transform("mean")
df["acima_media_dia"] = (df["percentual_gorjeta"] > df["media_do_dia"]).astype(int)
df[["dia", "percentual_gorjeta", "media_do_dia", "acima_media_dia"]].head().round(2)

,dia,percentual_gorjeta,media_do_dia,acima_media_dia
0,Sun,5.94,16.69,0
1,Sun,16.05,16.69,0
2,Sun,16.66,16.69,0
3,Sun,13.98,16.69,0
4,Sun,14.68,16.69,0


**Sua vez 6.1:** crie os atributos `media_do_turno` e `acima_media_turno`, seguindo o exemplo. Em seguida, conte quantas mesas de cada turno ficaram acima da média do próprio turno.

In [11]:
df['media_do_turno'] = df.groupby('turno')['percentual_gorjeta'].transform('mean')
df['acima_media_turno'] = (df['percentual_gorjeta'] > df['media_do_turno']).astype(int)
df.groupby('turno').agg(mesas=('acima_media_turno', 'size'), acima_da_media=('acima_media_turno', 'sum')).assign(percentual_acima=lambda x: x.acima_da_media / x.mesas * 100).round(2)

,mesas,acima_da_media,percentual_acima
turno,,,
Dinner,176,80,45.45
Lunch,67,30,44.78


## 7. Os novos atributos e as aulas anteriores
**Exemplo:** padronização do percentual da gorjeta para identificar mesas fora do padrão.

In [12]:
df["percentual_z"] = (df["percentual_gorjeta"] - df["percentual_gorjeta"].mean()) / df["percentual_gorjeta"].std()
df.nlargest(3, "percentual_z")[["conta", "gorjeta", "pessoas", "percentual_gorjeta", "percentual_z"]].round(2)

,conta,gorjeta,pessoas,percentual_gorjeta,percentual_z
172,7.25,5.15,2,71.03,8.98
178,9.60,4.00,2,41.67,4.18
67,3.07,1.00,1,32.57,2.69


**Sua vez 7.1:** calcule a correlação de `conta`, `pessoas` e `conta_por_pessoa` com a coluna `gorjeta` e, separadamente, com o atributo `percentual_gorjeta`.

In [13]:
variaveis = ['conta', 'pessoas', 'conta_por_pessoa']
alvos = ['gorjeta', 'percentual_gorjeta']
df[variaveis + alvos].corr(method='pearson').loc[variaveis, alvos].round(4)

,gorjeta,percentual_gorjeta
conta,0.6750,-0.3394
pessoas,0.4884,-0.1433
conta_por_pessoa,0.3465,-0.3145


**Pergunta 7.2:** a conta se relaciona com a gorjeta e com o percentual da gorjeta da mesma forma? O que isso significa para o dono do restaurante?

**Resposta:**

Não. A conta tem correlação positiva com a gorjeta em dólares (0,6750), mas negativa com o percentual da gorjeta (−0,3394). Assim, uma conta maior costuma acompanhar uma gorjeta absoluta maior, sem representar maior generosidade proporcional. Para organizar o atendimento, o dono deve distinguir receita em dólares e percentual, sem tratar a correlação como causa nem usar a amostra para julgar clientes individualmente.

## 8. Atributo próprio
**Sua vez 8.1:** crie um atributo novo que não tenha sido feito nesta aula. Escreva, na célula de resposta abaixo do código, a pergunta do dono do restaurante que ele responde e qual dos tipos vistos ele usa (razão, soma, indicador, faixa ou agregação).

In [14]:
df['gorjeta_por_pessoa'] = df['gorjeta'] / df['pessoas']
df.groupby('turno').agg(mesas=('gorjeta_por_pessoa', 'size'), gorjeta_media_por_pessoa=('gorjeta_por_pessoa', 'mean')).round(2)

,mesas,gorjeta_media_por_pessoa
turno,,
Dinner,176,1.23
Lunch,67,1.18


**Resposta:**

Pergunta: em qual turno a gorjeta recebida por pessoa atendida costuma ser maior? Criei `gorjeta_por_pessoa`, uma razão entre o valor da gorjeta e o número de pessoas. A média dessas razões compara mesas, atribuindo o mesmo peso a cada mesa; para calcular a razão global por cliente seria necessário dividir a soma das gorjetas pela soma das pessoas.

**Pergunta 8.2:** com base nos atributos criados, escreva uma recomendação de até três linhas para o dono do restaurante.

**Resposta:**

Compare gorjeta em dólares e percentual separadamente: grupos maiores deixam proporcionalmente menos nesta amostra.
Use gasto e gorjeta por pessoa para planejar o atendimento entre turnos.
Acompanhe mais semanas antes de mudar a escala; os dados são de um único garçom e não provam causalidade.

In [15]:
assert len(df) == 243
assert not df[['conta_por_pessoa', 'total_por_pessoa', 'gorjeta_por_pessoa']].isna().any().any()
df.to_csv('resultados/atributos_tips.csv', index=False)
print('Base original: 244 linhas. Após retirar uma duplicata: 243 linhas. CSV com atributos exportado.')

Base original: 244 linhas. Após retirar uma duplicata: 243 linhas. CSV com atributos exportado.
